In [5]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/nandand14/botsv3-enriched-features-behavioral/botsv3_enriched_features_behavioral.csv


In [6]:

import pandas as pd
import numpy as np

# ── Load ──────────────────────────────────────────────────────────────────────
csv_path = "/kaggle/input/datasets/nandand14/botsv3-enriched-features-behavioral/botsv3_enriched_features_behavioral.csv"
df = pd.read_csv(csv_path)

# ── Basic inspection ──────────────────────────────────────────────────────────
print("=" * 70)
print("SECTION 1 – SHAPE & OVERVIEW")
print("=" * 70)
print(f"Shape        : {df.shape[0]:,} rows  x  {df.shape[1]} columns")
print(f"\nAll columns ({len(df.columns)}):")
for i, col in enumerate(df.columns, 1):
    print(f"  {i:>3}. {col}")

print("\nData types:")
print(df.dtypes.to_string())

print("\nFirst 5 rows:")
print(df.head().to_string())

# ── Required columns check ────────────────────────────────────────────────────
print("\n" + "=" * 70)
print("SECTION 2 – REQUIRED COLUMNS PRESENCE CHECK")
print("=" * 70)

KEY_COLS   = ["user_name", "cloudtrail_time", "is_anomaly"]
FEAT_COLS  = [
    "hour", "day_of_week", "is_business_hours", "is_weekend",
    "time_since_last_change", "change_velocity", "change_magnitude",
    "rarity_score", "user_activity", "is_wildcard", "blast_radius",
    "privilege_direction", "is_error", "action_frequency",
    "resource_centrality", "neighbor_count", "clustering_coeff",
    "pagerank_score", "betweenness",
]
ALL_REQUIRED = KEY_COLS + FEAT_COLS

present   = [c for c in ALL_REQUIRED if c in df.columns]
missing   = [c for c in ALL_REQUIRED if c not in df.columns]

print(f"  Required columns  : {len(ALL_REQUIRED)}")
print(f"  ✅  Present        : {len(present)}")
print(f"  ❌  Missing        : {len(missing)}")
if missing:
    for m in missing:
        print(f"       – {m}")
else:
    print("  All 22 required columns found!")

# ── Missing value audit ───────────────────────────────────────────────────────
print("\n" + "=" * 70)
print("SECTION 3 – MISSING VALUES IN CRITICAL COLUMNS")
print("=" * 70)

null_series = {col: df[col].isnull().sum() for col in ALL_REQUIRED if col in df.columns}
null_report = pd.DataFrame([
    {"column": col, "missing_count": cnt, "missing_%": round(cnt / len(df) * 100, 3)}
    for col, cnt in null_series.items() if cnt > 0
])

if null_report.empty:
    print("  ✅  Zero missing values across all 22 required columns.")
else:
    print(f"  ⚠️  Columns with missing values ({len(null_report)}):")
    print(null_report.to_string(index=False))

# ── Unique users & rows-per-user ───────────────────────────────────────────────
print("\n" + "=" * 70)
print("SECTION 4 – USER-LEVEL STATISTICS")
print("=" * 70)

# Work with rows that have a valid user_name
df_named = df[df["user_name"].notna()].copy()
rows_per_user = df_named["user_name"].value_counts().sort_values(ascending=False)
n_users = rows_per_user.shape[0]
n_null_users = df["user_name"].isnull().sum()

print(f"  Total rows              : {df.shape[0]:,}")
print(f"  Rows with null user_name: {n_null_users:,}  ({n_null_users/len(df)*100:.1f}%)")
print(f"  Rows with valid user    : {len(df_named):,}")
print(f"  Unique named users      : {n_users:,}")
print(f"  Avg rows/user           : {rows_per_user.mean():.1f}")
print(f"  Min rows/user           : {rows_per_user.min()}")
print(f"  Max rows/user           : {rows_per_user.max():,}")
print(f"\n  Rows per user breakdown:")
print(rows_per_user.to_string())

# Users with < 10 rows
sparse_users = rows_per_user[rows_per_user < 10]
print(f"\n  Users with < 10 rows : {len(sparse_users):,}")
if len(sparse_users) > 0:
    print("  (These may be unsuitable for sliding-window modelling)")
    print(sparse_users.to_string())

# ── Feature dtype sanity ──────────────────────────────────────────────────────
print("\n" + "=" * 70)
print("SECTION 5 – FEATURE COLUMN DTYPE SUMMARY")
print("=" * 70)
existing_feats = [c for c in FEAT_COLS if c in df.columns]
print(df[existing_feats].dtypes.to_string())

# ── Sliding window verdict ────────────────────────────────────────────────────
print("\n" + "=" * 70)
print("SECTION 6 – SLIDING WINDOW VALIDITY VERDICT")
print("=" * 70)

all_cols_present  = len(missing) == 0
# user_name nulls are structural (service accounts / AWS internal); feature cols are clean
feat_nulls        = {c: df[c].isnull().sum() for c in FEAT_COLS if c in df.columns and df[c].isnull().sum() > 0}
no_feature_nulls  = len(feat_nulls) == 0
enough_data_users = (rows_per_user >= 10).sum()
pct_valid_users   = enough_data_users / n_users * 100 if n_users > 0 else 0
majority_valid    = pct_valid_users >= 50

print(f"  All 22 required columns present        : {'YES ✅' if all_cols_present else 'NO ❌'}")
print(f"  user_name null rows (likely svc accts) : {n_null_users:,} ({n_null_users/len(df)*100:.1f}%)")
print(f"  No missing values in feature columns   : {'YES ✅' if no_feature_nulls else 'NO ❌'  + str(feat_nulls)}")
print(f"  Unique named users                     : {n_users:,}")
print(f"  Users with ≥10 rows                    : {enough_data_users:,} / {n_users:,}  ({pct_valid_users:.0f}%)")
print(f"  Majority of users valid for windows    : {'YES ✅' if majority_valid else 'NO ❌'}")

# Named-user data is valid if cols present, features clean, majority of users have enough rows
verdict = all_cols_present and no_feature_nulls and majority_valid
print("\n" + "─" * 70)
print(f"  ➜  DATA VALID FOR SLIDING WINDOWS?  →  {'✅  YES' if verdict else '❌  NO'}")
if not verdict:
    if n_null_users > 0:
        print(f"     NOTE: {n_null_users:,} rows have null user_name (service/system accounts).")
        print(f"     ➜  After filtering to named users, the dataset IS valid.")
        print(f"     ➜  RECOMMENDATION: filter rows where user_name is not null before windowing.")
print("─" * 70)


SECTION 1 – SHAPE & OVERVIEW
Shape        : 6,571 rows  x  41 columns

All columns (41):
    1. hour
    2. day_of_week
    3. is_business_hours
    4. is_weekend
    5. time_since_last_change
    6. change_velocity
    7. change_magnitude
    8. rarity_score
    9. user_activity
   10. is_wildcard
   11. blast_radius
   12. privilege_direction
   13. is_error
   14. action_frequency
   15. resource_centrality
   16. neighbor_count
   17. clustering_coeff
   18. pagerank_score
   19. betweenness
   20. error_rate_recent
   21. cloudtrail_time
   22. event_name
   23. user_name
   24. source_ip
   25. net_bytes_total
   26. net_bytes_out
   27. net_unique_ips
   28. net_suspicious_ports
   29. net_exfil_score
   30. net_connections
   31. end_unique_processes
   32. end_suspicious_processes
   33. end_privileged_events
   34. end_logon_events
   35. end_process_creation
   36. is_anomaly
   37. anomaly_score
   38. triggered_rules
   39. confidence
   40. severity
   41. rule_count

Dat

In [7]:
import pandas as pd
import numpy as np
import json

# ── Constants ─────────────────────────────────────────────────────────────────
FEAT_COLS = [
    "hour", "day_of_week", "is_business_hours", "is_weekend",
    "time_since_last_change", "change_velocity", "change_magnitude",
    "rarity_score", "user_activity", "is_wildcard", "blast_radius",
    "privilege_direction", "is_error", "action_frequency",
    "resource_centrality", "neighbor_count", "clustering_coeff",
    "pagerank_score", "betweenness",
]
WINDOW_SIZE = 10
WINDOW_DURATION_MINUTES = 10
_WINDOW_DURATION = np.timedelta64(WINDOW_DURATION_MINUTES, "m")

# ══════════════════════════════════════════════════════════════════════════════
# STEP 1 – CLEAN THE DATAFRAME
# ══════════════════════════════════════════════════════════════════════════════
print("=" * 70)
print("STEP 1 – DATA CLEANING")
print("=" * 70)

_clean = df.copy()

# Parse cloudtrail_time as datetime (UTC, ignore errors -> NaT then drop)
_clean["cloudtrail_time"] = pd.to_datetime(_clean["cloudtrail_time"], utc=True, errors="coerce")
_n_nat = int(_clean["cloudtrail_time"].isna().sum())
if _n_nat > 0:
    print(f"  ⚠️  {_n_nat} rows with unparseable cloudtrail_time dropped.")
    _clean = _clean.dropna(subset=["cloudtrail_time"])
else:
    print("  ✅  cloudtrail_time parsed - zero NaT values.")

# Label column: session_label <- is_anomaly
_clean["session_label"] = _clean["is_anomaly"].astype(np.int32)

# Keep only rows with a real actor identity; do not merge nulls into one pseudo-user.
if "username" in _clean.columns:
    _id_col = "username"
elif "user_name" in _clean.columns:
    _id_col = "user_name"
else:
    raise KeyError("Expected one of: username or user_name")

_n_missing_identity = int(_clean[_id_col].isna().sum())
if _n_missing_identity > 0:
    _clean = _clean[_clean[_id_col].notna()].copy()
    print(f"  ⚠️  Dropped {_n_missing_identity:,} rows with null {_id_col} to avoid identity leakage.")
else:
    print(f"  ✅  {_id_col}: zero null values.")

_clean["username"] = _clean[_id_col].astype(str)

missing_feat_cols = [c for c in FEAT_COLS if c not in _clean.columns]
if missing_feat_cols:
    raise KeyError(f"Missing feature columns: {missing_feat_cols}")
if "event_name" not in _clean.columns:
    raise KeyError("Missing required column: event_name")

# Sort globally by cloudtrail_time
_clean = _clean.sort_values("cloudtrail_time").reset_index(drop=True)

# Drop full duplicates
_n_before_dedup = len(_clean)
_clean = _clean.drop_duplicates().reset_index(drop=True)
_n_dropped = _n_before_dedup - len(_clean)
print(f"  ✅  Duplicates dropped: {_n_dropped:,} rows removed.")
print(f"  ✅  Clean dataframe shape: {_clean.shape}")


# ══════════════════════════════════════════════════════════════════════════════
# STEP 2 – CAUSAL TIME-BASED SLIDING WINDOWS PER USER (O(n) OPTIMIZED)
# ══════════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 70)
print("STEP 2 – BUILDING CAUSAL TIME-BASED WINDOWS")
print("=" * 70)

X_list = []
y_list = []
event_name_list = []
window_end_time_list = []
user_window_counts = {}

_users = _clean["username"].unique()
_skipped_users = []

for _user in _users:
    # Group by username, sort chronologically
    _grp = _clean[_clean["username"] == _user].sort_values("cloudtrail_time").reset_index(drop=True)
    _n_rows = len(_grp)

    # Extract NumPy arrays for fast processing
    _features = _grp[FEAT_COLS].values.astype(np.float32)
    _labels = _grp["session_label"].values.astype(np.int32)
    _times = _grp["cloudtrail_time"].to_numpy(dtype="datetime64[ns]")
    _event_names_all = _grp["event_name"].astype(str).to_numpy()

    _user_windows = 0
    _left = 0  # Two-pointer boundary marker

    # Build sequences causally [t-10m, t]
    for _right in range(_n_rows):
        _window_end = _times[_right]
        _window_start = _window_end - _WINDOW_DURATION

        # Advance _left pointer until it enters our causal window timeframe
        while _left <= _right and _times[_left] <= _window_start:
            _left += 1
            
        # The true sequence is strictly bounded between _left and _right indices
        _win_feats = _features[_left : _right + 1]
        _win_labels = _labels[_left : _right + 1]
        _win_event_names = _event_names_all[_left : _right + 1].tolist()

        _k = len(_win_feats)
        if _k == 0:
            continue

        # Sequence is anomalous if any event seen so far in this window is anomalous.
        _win_label = int(np.max(_win_labels))

        # Pre-pad up to 10 timesteps, preserving most recent events at end of sequence
        if _k < WINDOW_SIZE:
            _pad = np.zeros((WINDOW_SIZE - _k, len(FEAT_COLS)), dtype=np.float32)
            _win_feats_fixed = np.vstack([_pad, _win_feats])
            _win_event_names = (["<PAD>"] * (WINDOW_SIZE - _k)) + _win_event_names
        else:
            _win_feats_fixed = _win_feats[-WINDOW_SIZE:]
            _win_event_names = _win_event_names[-WINDOW_SIZE:]

        X_list.append(_win_feats_fixed)
        y_list.append(_win_label)
        event_name_list.append(_win_event_names)
        window_end_time_list.append(_window_end)

        _user_windows += 1

    if _user_windows == 0:
        _skipped_users.append((_user, _n_rows))
    else:
        user_window_counts[_user] = _user_windows

if _skipped_users:
    print("  ⚠️  Users skipped (0 windows produced):")
    for _u, _r in _skipped_users:
        print(f"       {_u!r} -> {_r} rows")
else:
    print("  ✅  All users produced at least one window - no users skipped.")


# ══════════════════════════════════════════════════════════════════════════════
# STEP 3 – ASSEMBLE ARRAYS (GLOBAL CHRONOLOGICAL ORDER)
# ══════════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 70)
print("STEP 3 – ASSEMBLING ARRAYS")
print("=" * 70)

X_sequences = np.array(X_list, dtype=np.float32)                 # (N, 10, 19)
y_labels = np.array(y_list, dtype=np.int32)                      # (N,)
window_end_times = np.array(window_end_time_list, dtype="datetime64[ns]")

# Shape assertions
assert X_sequences.ndim == 3, f"Expected 3-D X, got {X_sequences.ndim}-D"
assert X_sequences.shape[1] == WINDOW_SIZE, f"Expected {WINDOW_SIZE} timesteps, got {X_sequences.shape[1]}"
assert X_sequences.shape[2] == len(FEAT_COLS), f"Expected {len(FEAT_COLS)} features, got {X_sequences.shape[2]}"
assert y_labels.ndim == 1, f"Expected 1-D y, got {y_labels.ndim}-D"
assert len(X_sequences) == len(y_labels), "X and y length mismatch"
assert len(event_name_list) == len(y_labels), "event_name_list length mismatch"
assert len(window_end_times) == len(y_labels), "window_end_times length mismatch"

# Global chronological sort so downstream split is truly temporal.
_order = np.argsort(window_end_times)
X_sequences = X_sequences[_order]
y_labels = y_labels[_order]
window_end_times = window_end_times[_order]
event_name_list = [event_name_list[i] for i in _order]

print(f"  X shape  : {X_sequences.shape}   (windows x timesteps x features)")
print(f"  y shape  : {y_labels.shape}      (windows,)")
print(f"  event_name_list length: {len(event_name_list):,}")
if len(window_end_times) > 0:
    print(f"  Window time range: {window_end_times[0]} -> {window_end_times[-1]}")

# Class breakdown
_n_anomaly = int((y_labels == 1).sum())
_n_normal = int((y_labels == 0).sum())
_total_w = len(y_labels)
_ratio = _n_anomaly / _n_normal if _n_normal > 0 else float("inf")
_anomaly_pct = (_n_anomaly / _total_w * 100.0) if _total_w > 0 else 0.0
_normal_pct = (_n_normal / _total_w * 100.0) if _total_w > 0 else 0.0

print(f"\n  Total windows  : {_total_w:,}")
print(f"  Anomaly (1)    : {_n_anomaly:,}  ({_anomaly_pct:.2f}%)")
print(f"  Normal  (0)    : {_n_normal:,}  ({_normal_pct:.2f}%)")
print(f"  Anomaly ratio  : {_ratio:.4f}  ({_ratio*100:.2f}% of normal)")

# Per-user window contribution (time-based grouping)
print("\n  Per-user window contribution (time-based grouping):")
for _u, _cnt in sorted(user_window_counts.items(), key=lambda x: -x[1]):
    print(f"    {_u!r:<35} -> {_cnt:,} windows")


# ══════════════════════════════════════════════════════════════════════════════
# STEP 4 – SAVE FILES
# ══════════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 70)
print("STEP 4 – SAVING FILES")
print("=" * 70)

np.save("X_sequences.npy", X_sequences)
np.save("y_labels.npy", y_labels)
# Save as int64 nanoseconds for robust cross-environment loading.
np.save("window_end_times.npy", window_end_times.astype(np.int64))
print("  ✅  X_sequences.npy saved.")
print("  ✅  y_labels.npy saved.")
print("  ✅  window_end_times.npy saved.")

with open("event_names_per_window.json", "w") as _f:
    json.dump(event_name_list, _f)
print("  ✅  event_names_per_window.json saved.")


# ══════════════════════════════════════════════════════════════════════════════
# STEP 5 – FINAL REPORT
# ══════════════════════════════════════════════════════════════════════════════
_lstm_ready = _total_w > 0 and X_sequences.shape[2] == len(FEAT_COLS)
_verdict_str = "✅  READY FOR LSTM TRAINING" if _lstm_ready else "❌  NOT READY"

if _ratio < 0.05:
    _imbalance_label = "SEVERE (consider oversampling / class weights)"
elif _ratio < 0.20:
    _imbalance_label = "MODERATE (class weights recommended)"
elif _ratio < 0.50:
    _imbalance_label = "MILD"
else:
    _imbalance_label = "BALANCED"

print("\n" + "=" * 70)
print("FINAL REPORT")
print("=" * 70)
print(f"  Data clean status     : ✅  {_clean.shape[0]:,} rows × {_clean.shape[1]} cols")
print(f"  Total windows created : {_total_w:,}")
print(f"  X_sequences shape     : {X_sequences.shape}  ✅")
print(f"  y_labels shape        : {y_labels.shape}  ✅")
print(f"  event_name_list       : {len(event_name_list):,} entries  ✅")
print(f"  Anomaly (1)           : {_n_anomaly:,}  ({_anomaly_pct:.2f}%)")
print(f"  Normal  (0)           : {_n_normal:,}  ({_normal_pct:.2f}%)")
print(f"  Class imbalance       : {_imbalance_label}")
print("  Labelling strategy    : causal lookback [t-10m, t]")
print(f"  LSTM readiness verdict: {_verdict_str}")
print("=" * 70)

STEP 1 – DATA CLEANING
  ✅  cloudtrail_time parsed - zero NaT values.
  ⚠️  Dropped 1,146 rows with null user_name to avoid identity leakage.
  ✅  Duplicates dropped: 2,794 rows removed.
  ✅  Clean dataframe shape: (2631, 43)

STEP 2 – BUILDING CAUSAL TIME-BASED WINDOWS
  ✅  All users produced at least one window - no users skipped.

STEP 3 – ASSEMBLING ARRAYS
  X shape  : (2631, 10, 19)   (windows x timesteps x features)
  y shape  : (2631,)      (windows,)
  event_name_list length: 2,631
  Window time range: 2018-08-20T09:01:54.000000000 -> 2018-08-20T15:15:04.000000000

  Total windows  : 2,631
  Anomaly (1)    : 1,946  (73.96%)
  Normal  (0)    : 685  (26.04%)
  Anomaly ratio  : 2.8409  (284.09% of normal)

  Per-user window contribution (time-based grouping):
    'splunk_access'                     -> 1,503 windows
    'web_admin'                         -> 567 windows
    'bstoll'                            -> 492 windows
    'btun'                              -> 69 windows

STE

In [8]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

# ─────────────────────────────────────────────
# 0.  Helpers (leakage-safe temporal split)
# ─────────────────────────────────────────────

def chronological_split_with_embargo(window_end_times_ns, val_ratio=0.20, test_ratio=0.20, embargo_minutes=10):
    """Chronological split with embargo gap to reduce overlap leakage between splits."""
    if val_ratio <= 0 or test_ratio <= 0 or (val_ratio + test_ratio) >= 1:
        raise ValueError("val_ratio and test_ratio must be > 0 and sum to < 1")

    n = len(window_end_times_ns)
    if n < 10:
        raise ValueError("Not enough samples for train/val/test split")

    n_test = max(1, int(np.round(n * test_ratio)))
    n_val = max(1, int(np.round(n * val_ratio)))
    n_train = n - n_val - n_test
    if n_train < 1:
        raise ValueError("Split produced empty train set")

    train_end_ns = window_end_times_ns[n_train - 1]
    val_end_ns = window_end_times_ns[n_train + n_val - 1]
    embargo_ns = np.int64(embargo_minutes * 60 * 1_000_000_000)

    train_idx = np.where(window_end_times_ns <= train_end_ns)[0]
    val_idx = np.where((window_end_times_ns > (train_end_ns + embargo_ns)) & (window_end_times_ns <= val_end_ns))[0]
    test_idx = np.where(window_end_times_ns > (val_end_ns + embargo_ns))[0]

    used_embargo = True
    if len(val_idx) == 0 or len(test_idx) == 0:
        # Fallback if timeline is too dense for strict embargo boundaries.
        train_idx = np.arange(0, n_train)
        val_idx = np.arange(n_train, n_train + n_val)
        test_idx = np.arange(n_train + n_val, n)
        used_embargo = False

    meta = {
        "used_embargo": used_embargo,
        "embargo_minutes": embargo_minutes,
        "train_end_ns": int(train_end_ns),
        "val_end_ns": int(val_end_ns),
    }
    return train_idx, val_idx, test_idx, meta


def compute_f1(y_true, y_pred_bin):
    tp = int(((y_pred_bin == 1) & (y_true == 1)).sum())
    fp = int(((y_pred_bin == 1) & (y_true == 0)).sum())
    fn = int(((y_pred_bin == 0) & (y_true == 1)).sum())
    prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    rec = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    return 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0.0, prec, rec


def compute_auc_roc(y_true, y_scores):
    """Efficient trapezoidal AUC-ROC via numpy sort (no sklearn)."""
    pos = int(y_true.sum())
    neg = int(len(y_true) - pos)
    if pos == 0 or neg == 0:
        return 0.5

    order = np.argsort(-y_scores)
    y_sorted = y_true[order]

    tpr, fpr, cum_tp, cum_fp = [0.0], [0.0], 0, 0
    for label in y_sorted:
        if label == 1:
            cum_tp += 1
        else:
            cum_fp += 1
        tpr.append(cum_tp / pos)
        fpr.append(cum_fp / neg)
    tpr.append(1.0)
    fpr.append(1.0)
    return float(np.trapezoid(tpr, fpr))


def confusion_matrix_np(y_true, y_pred):
    tp = int(((y_pred == 1) & (y_true == 1)).sum())
    tn = int(((y_pred == 0) & (y_true == 0)).sum())
    fp = int(((y_pred == 1) & (y_true == 0)).sum())
    fn = int(((y_pred == 0) & (y_true == 1)).sum())
    return np.array([[tn, fp], [fn, tp]])


def _fmt_time(ns_value):
    return np.datetime_as_string(np.datetime64(int(ns_value), "ns"), unit="s")


# ─────────────────────────────────────────────
# 1. Load Data
# ─────────────────────────────────────────────
X_sequences = np.load("X_sequences.npy").astype(np.float32)   # (N, 10, 19)
y_labels = np.load("y_labels.npy").astype(np.float32)         # (N,)
window_end_times_ns = np.load("window_end_times.npy").astype(np.int64)

if not (len(X_sequences) == len(y_labels) == len(window_end_times_ns)):
    raise ValueError("X, y, and window_end_times length mismatch")

# Defensive sort to guarantee strict chronology before splitting.
_order = np.argsort(window_end_times_ns)
X_sequences = X_sequences[_order]
y_labels = y_labels[_order]
window_end_times_ns = window_end_times_ns[_order]

print(f"Loaded  X: {X_sequences.shape},  y: {y_labels.shape}")
print(f"Class distribution — Anomaly: {y_labels.mean()*100:.1f}%  |  Normal: {(1-y_labels).mean()*100:.1f}%")
print(f"Window time range — start: {_fmt_time(window_end_times_ns[0])}  end: {_fmt_time(window_end_times_ns[-1])}")

# ─────────────────────────────────────────────
# 2. Chronological 60 / 20 / 20 Split (+ embargo)
# ─────────────────────────────────────────────
train_idx, val_idx, test_idx, split_meta = chronological_split_with_embargo(
    window_end_times_ns,
    val_ratio=0.20,
    test_ratio=0.20,
    embargo_minutes=10,
)

X_train, y_train = X_sequences[train_idx], y_labels[train_idx]
X_val, y_val = X_sequences[val_idx], y_labels[val_idx]
X_test, y_test = X_sequences[test_idx], y_labels[test_idx]

print(f"\nSplit sizes — Train: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test)}")
print(f"Embargo applied: {'YES' if split_meta['used_embargo'] else 'NO (fallback split)'}")
print(f"Train end time: {_fmt_time(split_meta['train_end_ns'])}")
print(f"Val end time  : {_fmt_time(split_meta['val_end_ns'])}")

for split_name, y_s in [("Train", y_train), ("Val", y_val), ("Test", y_test)]:
    anomaly_pct = y_s.mean() * 100 if len(y_s) > 0 else 0.0
    print(f"  {split_name}: anomaly={anomaly_pct:.1f}%  normal={100-anomaly_pct:.1f}%")

# ─────────────────────────────────────────────
# 3. DataLoaders
# ─────────────────────────────────────────────
def make_loader(X, y, batch_size=64, shuffle=True):
    ds = TensorDataset(torch.from_numpy(X), torch.from_numpy(y))
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle)

train_loader = make_loader(X_train, y_train, shuffle=True)
val_loader = make_loader(X_val, y_val, shuffle=False)
test_loader = make_loader(X_test, y_test, shuffle=False)

# ─────────────────────────────────────────────
# 4. LSTM Model
# ─────────────────────────────────────────────
class LSTMAnomalyDetector(nn.Module):
    def __init__(self, input_size=19, hidden_size=64, num_layers=2, dropout=0.3):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            dropout=dropout,
            batch_first=True,
        )
        self.fc = nn.Linear(hidden_size, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        lstm_out, _ = self.lstm(x)           # (batch, seq_len, hidden)
        last_hidden = lstm_out[:, -1, :]     # last time step
        out = self.fc(last_hidden)           # (batch, 1)
        return self.sigmoid(out).squeeze(1)  # (batch,) -> P_seq per sample


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"\nUsing device: {device}")

lstm_model = LSTMAnomalyDetector(input_size=19, hidden_size=64, num_layers=2, dropout=0.3).to(device)
print(lstm_model)
total_params = sum(p.numel() for p in lstm_model.parameters() if p.requires_grad)
print(f"Trainable parameters: {total_params:,}")

# ─────────────────────────────────────────────
# 5. Class-Weighted BCELoss (train set only)
# ─────────────────────────────────────────────
_n_total_train = len(y_train)
_n_anomaly_train = int((y_train == 1).sum())
_n_normal_train = int((y_train == 0).sum())

if _n_anomaly_train == 0 or _n_normal_train == 0:
    pos_weight_val = 1.0
    print("\n⚠️  Train split has a single class; using pos_weight_val = 1.0")
else:
    pos_weight_val = _n_normal_train / _n_anomaly_train

anomaly_pct_train = (_n_anomaly_train / _n_total_train * 100.0) if _n_total_train > 0 else 0.0
normal_pct_train = (_n_normal_train / _n_total_train * 100.0) if _n_total_train > 0 else 0.0

print("\nClass distribution from TRAIN split only:")
print(f"  Anomaly (1): {_n_anomaly_train:,}  ({anomaly_pct_train:.2f}%)")
print(f"  Normal  (0): {_n_normal_train:,}  ({normal_pct_train:.2f}%)")
print(f"  Computed pos_weight_val = {pos_weight_val:.4f}")

bce_base = nn.BCELoss(reduction="none")


def weighted_bce(preds, targets, pos_weight=pos_weight_val):
    losses = bce_base(preds, targets)
    weights = torch.where(
        targets == 1,
        torch.full_like(targets, pos_weight),
        torch.ones_like(targets),
    )
    return (losses * weights).mean()


optimizer = torch.optim.Adam(lstm_model.parameters(), lr=1e-3)

# ─────────────────────────────────────────────
# 6. Training + Validation Loop
# ─────────────────────────────────────────────
def evaluate_loader(loader):
    lstm_model.eval()
    all_preds, all_targets = [], []
    total_loss = 0.0
    with torch.no_grad():
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            preds = lstm_model(xb)
            loss = weighted_bce(preds, yb)
            total_loss += loss.item() * len(yb)
            all_preds.extend(preds.cpu().numpy())
            all_targets.extend(yb.cpu().numpy())

    avg_loss = total_loss / len(loader.dataset)
    all_preds = np.array(all_preds)
    all_targets = np.array(all_targets)
    bin_preds = (all_preds >= 0.5).astype(int)
    f1_val, _, _ = compute_f1(all_targets, bin_preds)
    auc_val = compute_auc_roc(all_targets, all_preds)
    return avg_loss, f1_val, auc_val


EPOCHS = 30
print(f"\n{'='*70}")
print(f"{'Epoch':>6}  {'Train Loss':>11}  {'Val Loss':>9}  {'Val F1':>7}  {'Val AUC':>8}")
print(f"{'─'*70}")

lstm_train_history = []

for epoch in range(1, EPOCHS + 1):
    lstm_model.train()
    epoch_loss = 0.0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        preds = lstm_model(xb)
        loss = weighted_bce(preds, yb)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item() * len(yb)

    train_loss_epoch = epoch_loss / len(train_loader.dataset)
    val_loss, val_f1, val_auc = evaluate_loader(val_loader)

    lstm_train_history.append({
        "epoch": epoch,
        "train_loss": train_loss_epoch,
        "val_loss": val_loss,
        "val_f1": val_f1,
        "val_auc": val_auc,
    })
    print(f"{epoch:>6d}  {train_loss_epoch:>11.5f}  {val_loss:>9.5f}  {val_f1:>7.4f}  {val_auc:>8.4f}")

print(f"{'='*70}")

# ─────────────────────────────────────────────
# 7. Final Test Set Evaluation
# ─────────────────────────────────────────────
lstm_model.eval()
test_preds_list, test_targets_list = [], []
test_total_loss = 0.0
with torch.no_grad():
    for xb, yb in test_loader:
        xb, yb = xb.to(device), yb.to(device)
        preds = lstm_model(xb)
        loss = weighted_bce(preds, yb)
        test_total_loss += loss.item() * len(yb)
        test_preds_list.extend(preds.cpu().numpy())
        test_targets_list.extend(yb.cpu().numpy())

lstm_test_loss = test_total_loss / len(test_loader.dataset)
test_preds_arr = np.array(test_preds_list)
test_targets_arr = np.array(test_targets_list)
test_bin_preds = (test_preds_arr >= 0.5).astype(int)

lstm_test_f1, test_prec, test_rec = compute_f1(test_targets_arr, test_bin_preds)
lstm_test_auc = compute_auc_roc(test_targets_arr, test_preds_arr)
lstm_test_cm = confusion_matrix_np(test_targets_arr, test_bin_preds)

# Classification report
tn, fp, fn, tp = lstm_test_cm[0, 0], lstm_test_cm[0, 1], lstm_test_cm[1, 0], lstm_test_cm[1, 1]
n_pos = int(test_targets_arr.sum())
n_neg = int((1 - test_targets_arr).sum())
acc = (tp + tn) / len(test_targets_arr)

print(f"\n{'━'*70}")
print("  FINAL TEST SET RESULTS")
print(f"{'━'*70}")
print(f"  Test Loss    : {lstm_test_loss:.5f}")
print(f"  Test F1      : {lstm_test_f1:.4f}")
print(f"  Test AUC-ROC : {lstm_test_auc:.4f}")
print(f"  Test Accuracy: {acc:.4f}")
print(f"  Precision    : {test_prec:.4f}  |  Recall: {test_rec:.4f}")

print("\nConfusion Matrix (rows=actual, cols=predicted):")
print("                  Pred Normal  Pred Anomaly")
print(f"  Actual Normal     {tn:>7}       {fp:>7}")
print(f"  Actual Anomaly    {fn:>7}       {tp:>7}")

print("\nClassification Report:")
print(f"{'─'*55}")
print(f"  {'Class':<14} {'Prec':>7} {'Rec':>7} {'F1':>7} {'Support':>9}")
prec_n = tn / (tn + fn) if (tn + fn) > 0 else 0.0
rec_n = tn / (tn + fp) if (tn + fp) > 0 else 0.0
f1_n = 2 * prec_n * rec_n / (prec_n + rec_n) if (prec_n + rec_n) > 0 else 0.0
print(f"  {'Normal':<14} {prec_n:>7.4f} {rec_n:>7.4f} {f1_n:>7.4f} {n_neg:>9}")
print(f"  {'Anomaly':<14} {test_prec:>7.4f} {test_rec:>7.4f} {lstm_test_f1:>7.4f} {n_pos:>9}")
print(f"{'─'*55}")
print(f"  {'Accuracy':<14} {'':>7} {'':>7} {acc:>7.4f} {len(test_targets_arr):>9}")
print(f"{'━'*70}")

# ─────────────────────────────────────────────
# 8. Save Model
# ─────────────────────────────────────────────
torch.save(lstm_model.state_dict(), "lstm_model.pt")
print("\n✅  lstm_model.pt saved successfully.")

Loaded  X: (2631, 10, 19),  y: (2631,)
Class distribution — Anomaly: 74.0%  |  Normal: 26.0%
Window time range — start: 2018-08-20T09:01:54  end: 2018-08-20T15:15:04

Split sizes — Train: 1587 | Val: 371 | Test: 416
Embargo applied: YES
Train end time: 2018-08-20T13:10:35
Val end time  : 2018-08-20T13:57:31
  Train: anomaly=84.4%  normal=15.6%
  Val: anomaly=49.6%  normal=50.4%
  Test: anomaly=46.4%  normal=53.6%

Using device: cpu
LSTMAnomalyDetector(
  (lstm): LSTM(19, 64, num_layers=2, batch_first=True, dropout=0.3)
  (fc): Linear(in_features=64, out_features=1, bias=True)
  (sigmoid): Sigmoid()
)
Trainable parameters: 55,105

Class distribution from TRAIN split only:
  Anomaly (1): 1,340  (84.44%)
  Normal  (0): 247  (15.56%)
  Computed pos_weight_val = 0.1843

 Epoch   Train Loss   Val Loss   Val F1   Val AUC
──────────────────────────────────────────────────────────────────────
     1      0.18460    0.70766   0.6919    0.7474
     2      0.13271    1.05460   0.7025    0.7092
   